In [0]:
%sql
use catalog databricks_project;
use schema schema

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_staging AS
SELECT
    customer_id,
    name,
    contact.email AS email,
    region,
    md5(concat_ws('||', contact.email, region)) AS hash_value
FROM bronze_customers;

In [0]:
%sql
select * from customers_staging

customer_id,name,email,region,hash_value
CUST001,John Doe,john@example.com,West,2085d22a2e770f261a404b8fecf76e2b
CUST002,Jane Smith,jane@example.com,West,b811a5f2b61dbefe2e09ab686a26a103
CUST003,Mike Ross,mike@example.com,South,3820f6e48799613b0f7e49cc30ba4e04
CUST004,Rachel Zane,rachel@example.com,West,c0d09f33955b4b0c66e757b2b72e3629
CUST005,Harvey Specter,harvey@example.com,East,3d30b473e02d2a796ccc3227c45670d7
CUST006,Donna Paulsen,donna@example.com,North,faec22fd4b3f3f710c490724bf8000c4
CUST007,Karthika,k@gmail.com,south,b329f307ad23214bfb48da83af1a7d17
CUST008,guest1,g@gmail.com,North,f94f4450ae4f6f93e401f02da7b2fec7


In [0]:
%sql
CREATE TABLE if not exists silver_customers_scd2 (
  customer_key BIGINT,
  customer_id STRING,
  name STRING,
  email STRING,
  region STRING,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  is_current BOOLEAN,
  hash_value STRING
);

In [0]:
%sql
MERGE INTO silver_customers_scd2 AS target
USING customers_staging AS source
ON target.customer_id = source.customer_id
AND target.is_current = TRUE
WHEN MATCHED AND target.hash_value != source.hash_value
THEN UPDATE SET
    target.is_current = FALSE,
    target.end_date = current_timestamp();

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
INSERT INTO silver_customers_scd2
SELECT
    uuid() AS customer_key,
    source.customer_id,
    source.name,
    source.email,
    source.region,
    current_timestamp() AS start_date,
    NULL AS end_date,
    TRUE AS is_current,
    source.hash_value
FROM customers_staging source
LEFT JOIN silver_customers_scd2 target
ON source.customer_id = target.customer_id
AND target.is_current = TRUE
WHERE target.customer_id IS NULL
   OR target.hash_value != source.hash_value;

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
select * from databricks_project.schema.silver_customers_scd2
order by customer_id

customer_key,customer_id,name,email,region,start_date,end_date,is_current,hash_value
0,CUST001,John Doe,john@example.com,West,2026-04-09T07:19:38.103Z,null,true,2085d22a2e770f261a404b8fecf76e2b
1,CUST002,Jane Smith,jane@example.com,West,2026-04-09T07:19:38.103Z,null,true,b811a5f2b61dbefe2e09ab686a26a103
2,CUST003,Mike Ross,mike@example.com,South,2026-04-09T07:19:38.103Z,null,true,3820f6e48799613b0f7e49cc30ba4e04
3,CUST004,Rachel Zane,rachel@example.com,West,2026-04-09T07:19:38.103Z,null,true,c0d09f33955b4b0c66e757b2b72e3629
4,CUST005,Harvey Specter,harvey@example.com,East,2026-04-09T07:19:38.103Z,null,true,3d30b473e02d2a796ccc3227c45670d7
5,CUST006,Donna Paulsen,donna@example.com,North,2026-04-09T07:19:38.103Z,null,true,faec22fd4b3f3f710c490724bf8000c4
6,CUST007,Karthika,k@gmail.com,south,2026-04-09T07:19:38.103Z,null,true,b329f307ad23214bfb48da83af1a7d17
7,CUST008,guest1,gtudc@gmail.com,North,2026-04-09T07:19:38.103Z,2026-04-09T07:26:31.159Z,false,66b9e663b9c4c610cebac790f1e769ad
0,CUST008,guest1,g@gmail.com,North,2026-04-09T07:26:36.453Z,null,true,f94f4450ae4f6f93e401f02da7b2fec7


In [0]:
%sql
-- drop table databricks_project.schema.silver_customers_scd2

In [0]:
%sql
select * from databricks_project.schema.silver_customers_scd2

customer_key,customer_id,name,email,region,start_date,end_date,is_current,hash_value
0,CUST001,John Doe,john@example.com,West,2026-04-09T07:19:38.103Z,null,true,2085d22a2e770f261a404b8fecf76e2b
1,CUST002,Jane Smith,jane@example.com,West,2026-04-09T07:19:38.103Z,null,true,b811a5f2b61dbefe2e09ab686a26a103
2,CUST003,Mike Ross,mike@example.com,South,2026-04-09T07:19:38.103Z,null,true,3820f6e48799613b0f7e49cc30ba4e04
3,CUST004,Rachel Zane,rachel@example.com,West,2026-04-09T07:19:38.103Z,null,true,c0d09f33955b4b0c66e757b2b72e3629
4,CUST005,Harvey Specter,harvey@example.com,East,2026-04-09T07:19:38.103Z,null,true,3d30b473e02d2a796ccc3227c45670d7
5,CUST006,Donna Paulsen,donna@example.com,North,2026-04-09T07:19:38.103Z,null,true,faec22fd4b3f3f710c490724bf8000c4
6,CUST007,Karthika,k@gmail.com,south,2026-04-09T07:19:38.103Z,null,true,b329f307ad23214bfb48da83af1a7d17
7,CUST008,guest1,gtudc@gmail.com,North,2026-04-09T07:19:38.103Z,2026-04-09T07:26:31.159Z,false,66b9e663b9c4c610cebac790f1e769ad
0,CUST008,guest1,g@gmail.com,North,2026-04-09T07:26:36.453Z,null,true,f94f4450ae4f6f93e401f02da7b2fec7
